# Run pipeline: all fires

Clean, repeatable batch runner. Fire selection/exploration/diagnostics stay in
`data.ipynb`; this notebook just runs `build_cell_table` over a defined list of
fires and saves the pooled result. Re-running is cheap once biomass/dNBR/forest-type
are cached locally (only new fires trigger a real Earth Engine wait).

Known issues that still apply to everything produced here (see README):
- GLAD height unreliable on steep terrain (#8) - exclude `<10m` from conclusions.
- CCI biomass sensitivity varies a lot by fire for unexplained reasons (Section 10).
- Fire Atlas `landcover` label does not reliably predict pixel-level forest type
  (Section 12) - always check `forest_type_label`, never the Atlas label.


In [ ]:
import sys
sys.path.append("..")

import ee
ee.Authenticate()   # skip if already authenticated this session
ee.Initialize(project="tree-fire-510209")

import pandas as pd
from src.pipeline import build_cell_table
from src.fires import utm_epsg_for


## Fire configs

One entry per fire run so far (11 total): 4 NAM (Evergreen Needleleaf), 4 SE Asia
(mostly Evergreen Broadleaf), 3 new biomes (southern Africa, Amazon, Australia).
UTM zone is computed from each fire's coordinates rather than hardcoded.


In [ ]:
fire_configs = [
    # NAM
    dict(uid="2020_112", lon=-122.782, lat=39.6938,
         pre_fire_window=("2020-07-01", "2020-08-10"), post_fire_window=("2021-07-01", "2021-08-31"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2020_135", lon=-121.094, lat=39.8396,
         pre_fire_window=("2020-07-01", "2020-08-15"), post_fire_window=("2021-07-01", "2021-08-31"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2020_464", lon=-122.128, lat=37.0396,
         pre_fire_window=("2020-07-01", "2020-08-10"), post_fire_window=("2021-07-01", "2021-08-31"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2021_190", lon=-120.556, lat=38.5896,
         pre_fire_window=("2021-06-15", "2021-08-10"), post_fire_window=("2022-06-15", "2022-08-31"),
         agb_before_year=2020, agb_after_years=[2022]),
    # SE Asia
    dict(uid="2020_841874", lon=101.5797, lat=18.8730,
         pre_fire_window=("2020-01-01", "2020-03-15"), post_fire_window=("2021-01-01", "2021-03-15"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2020_838262", lon=97.6276, lat=18.9896,
         pre_fire_window=("2019-11-01", "2020-01-20"), post_fire_window=("2020-11-01", "2021-01-20"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2020_818141", lon=94.8991, lat=24.6730,
         pre_fire_window=("2020-01-01", "2020-03-15"), post_fire_window=("2021-01-01", "2021-03-15"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2021_750951", lon=98.9752, lat=20.9855,
         pre_fire_window=("2021-01-01", "2021-03-10"), post_fire_window=("2022-01-01", "2022-03-10"),
         agb_before_year=2020, agb_after_years=[2022]),
    # new biomes
    dict(uid="2020_325095", lon=19.8160, lat=-7.3770,
         pre_fire_window=("2020-04-15", "2020-06-15"), post_fire_window=("2021-04-15", "2021-06-15"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2020_66789", lon=-52.0190, lat=-8.1812,
         pre_fire_window=("2020-06-01", "2020-08-15"), post_fire_window=("2021-06-01", "2021-08-15"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
    dict(uid="2020_901450", lon=153.3236, lat=-25.0645,
         pre_fire_window=("2020-08-01", "2020-10-10"), post_fire_window=("2021-08-01", "2021-10-10"),
         agb_before_year=2019, agb_after_years=[2021, 2022]),
]
print(len(fire_configs), "fires configured")


In [ ]:
results = {}
for cfg in fire_configs:
    uid = cfg["uid"]
    lon, lat = cfg.pop("lon"), cfg.pop("lat")
    print("running", uid)
    try:
        results[uid] = build_cell_table(utm_epsg=utm_epsg_for(lon, lat), **cfg)
        print(uid, "done:", len(results[uid]), "cells")
    except Exception as e:
        print(uid, "FAILED:", e)
    cfg["lon"], cfg["lat"] = lon, lat   # put back in case this cell is re-run


## Pool, check forest-type recovery, save

Compares the `unknown` share using the new two-band fallback (`forest_type_label`,
computed inside `build_cell_table`) against the old single-band approach, to see
how much the `discrete_classification` fallback actually recovered.


In [ ]:
full = pd.concat([df.assign(uid=uid) for uid, df in results.items()], ignore_index=True)
full["height_class"] = pd.cut(full["height"], [0, 10, 20, 100], labels=["<10m", "10-20m", ">20m"])

print("fires:", full["uid"].nunique(), " total cells:", len(full))
print(full["forest_type_label"].value_counts(normalize=True))

# old (primary-band-only) unknown share, for comparison, using the raw forest_type codes
from src.forest_type import FOREST_TYPE_LABELS
old_label = full["forest_type"].map(FOREST_TYPE_LABELS)
print("old unknown share (forest_type band only):", (old_label == "unknown").mean())
print("new unknown share (with discrete_classification fallback):", (full["forest_type_label"] == "unknown").mean())


In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
full.to_parquet("../data/processed/all_cells.parquet")
print("saved", len(full), "cells to ../data/processed/all_cells.parquet")


## Curve fits (draft, not final - uncertainty still wide, see README Section 12)

In [ ]:
from src.curves import fit_all_groups, bootstrap_curve_band

fit_table = fit_all_groups(full)
print(fit_table)
